### Optical water type
*** this code is elegant and done 'the right way' but consumes too much memory!

Characterise lakes according to one of 13 types (can assume that ours are not clear/oligotrophic) by comparing to a reference spectrum (another great use of the geomedian!). This is a spectral angle test. Methods as per Spyrakos et al 2018. 

- code here caters for msi_agm and oli_agm sensors (ie., the annual geomedian)
- easily adapted to msi and oli through dictionary entries
- the optical water type (OWT) is estimated from the spectral angle between the pixel and the refence spectra for the 13 OWTs
- extension to other sensors (TM, OLCI) also quite straightforward but the OWT refence data would need to be brought in.

- the OWT can be estimated:
- - pixel by pixel (implemented, this is the hard one)
  - from the median pixel value for each year (implemented) 
  - from the median pixel values for each pixel (could be implemented)
  - from the median pixel value for the waterbody through all time (could be implemented)

we can do this for Landsat and for Sentinel, however in the geomedian 
- OLI has only 3 available bands (2,3,4), missing band 1
- MSI has 6  (of 7 useful bands, missing band 1) 
- TM  has 4  (all of 1,2,3,4)



**To do, ideally :**
**Upload the full reference spectra and establish an import function to bring in as a data array.**
- Here, I have estimated the mean band reflectance for each sensor in excell using the spectral response curves (MSI response functions are available on-line) and used the results directly 

### Grouping of OWTs as per Zhang et al.:
- oligotrophic (clear)           : 3, 9, 13           (apply NASA OCx model for ChlA)
- eutopic and bluegreen waters   : 1, 2, 4, 5, 11, 12 (apply two band models of Gurlin et al for ChlA)
- hypereutrophic and green-brown : 6, 7, 8, 10        (apply TC2 alg for ChlA)


In [724]:
def OWT_deprecated(ds,instrument):
    #this code will need revisiting to support non-geomedian data which has more bands available. 
    #also, right now, msi and oli are dealt with separately rather than as alternatives...
    # it would be nice to remove this 'hard-wiring'.
    
    print('Determining the Optical Water Type ...')
    owt_groups = {}
    owt_groups['oligotrophic']                   = [3,9,13]
    owt_groups['eutropic and blue-green']        = [1,2,4,5,11,12]
    owt_groups['hypereutrophic and green-brown'] = [6,7,8,10]

    # estimated spectra for each optical water type for each sensor (MSI, OLI, TM) calculated from full spectra table provided by vagelis spyrakos
    #columns are bands, rows are OWT
    #1	2	3	4	5	6	7
    
    owt_data_msi = np.asarray ([0.000518281,0.002873764,0.003495403,0.001515508,0.002204739,0.003842756,0.005026098,
                    0.001310923,0.007407241,0.007373993,0.003239791,0.001309939,0.000351881,0.00044227,               
        0.002720793,0.012390417,0.00737146,0.001458816,0.000422552,0.000142766,0.00018326,
        0.001011837,0.006135417,0.006416972,0.00401114,0.001825363,0.000454228,0.000554565,
        0.001368776,0.005729538,0.004259349,0.00409354,0.001965091,0.001276867,0.001526747,
        0.000947881,0.005756157,0.006822492,0.003394953,0.002302141,0.000694368,0.000822148,
        0.000720924,0.003734303,0.004729343,0.002251117,0.003381287,0.002206305,0.002708644,
        0.000830314,0.004805815,0.005933108,0.003108917,0.002882319,0.001181092,0.001454993,
        0.001877709,0.009255961,0.007552662,0.002441745,0.000847245,0.000279124,0.000375539,
        0.000842475,0.00240552,0.002787709,0.005383652,0.003098927,0.001542782,0.002194582,
        0.000746178,0.004633521,0.005087786,0.00500395,0.002445958,0.000762379,0.000962499,
        0.001439293,0.006599122,0.005582806,0.003590672,0.001796926,0.000768249,0.000934859,
        0.006322908,0.014258851,0.002314198,0.000275429,8.56521E-05,6.41172E-05,0.000110569]).reshape(13,7)

    #	oli bands:1,2,3,4, rows are owt:
    owt_data_oli = np.asarray ([0.000536567,0.000860402,0.002160328,0.001178919,
         0.001346095,0.002267667,0.004757344,0.002459032,
         0.002818286,0.0041035,0.004739951,0.001088903,
         0.001034619,0.001850067,0.004198279,0.003021194,
         0.001429095,0.001886083,0.002822574,0.00284571,
         0.000971,0.001697917,0.004364393,0.002620484,
         0.000754857,0.001114315,0.002976344,0.001750548,
         0.000856452,0.00142315,0.003775541,0.002414677,
         0.001945476,0.002931367,0.004860984,0.001833516,
         0.000936538,0.000772627,0.001869377,0.003541968,
         0.000764871,0.001385817,0.00335777,0.003580806,
         0.001499333,0.002115267,0.003644426,0.00264,
         0.006800333,0.00588125,0.001539375,0.000199513]).reshape(13,4)

    owt_list = ['owt1','owt2','owt3','owt4','owt5','owt6','owt7','owt8','owt9','owt10','owt11','owt12','owt13']
    owt_msi = xr.DataArray(owt_data_msi,
                           dims=("owt","band"),
                           coords={"owt": owt_list, "band": np.arange(1,8,1)},
                           attrs = {"desc" : "optical water types - characteristic reflectances for msi"})
    owt_oli = xr.DataArray(owt_data_oli,
                    dims=("owt","band"),
                    coords={"owt": owt_list, "band":np.arange(1,5,1)},
                    attrs = {"desc" : "owt reflectances for oli"})
    OWT = {'msi_agm': owt_msi,'oli_agm':owt_oli}

    #owt types are stored in their own dimension to support vector multiplication

    #first, analysis per cell 
    # build a working dataset with the msi and oli values stored in their own dimension

    shortname = instrument[0:3]
    bandlists = {"msi_agm" : {"bands" : ['msi02','msi03','msi04','msi05','msi06','msi07'],"suffix": '_agmr',"dim_name": 'msi_band',"var_name":'msi_vals',"band_index": [1,7]},
                 "oli_agm" : {"bands" : ['oli02','oli03','oli04'                        ],"suffix": '_agmr',"dim_name": 'oli_band',"var_name":'oli_vals',"band_index": [1,4]}}    
    bandlist = bandlists[instrument]['bands'];    suffix = bandlists[instrument]['suffix'];     dim_name = bandlists[instrument]['dim_name']; 
    var_name = bandlists[instrument]['var_name']; i,j    = bandlists[instrument]['band_index'] 
    owt_data = np.asarray(OWT[instrument])
    varlist =      ['watermask']
    for name in bandlist: 
        varlist = np.append(varlist,str(name)+suffix)
    mydataset = ds[varlist]

    #create a dimension for the surface reflectance data
    #this part is long-hand because I don't know how to get it to iterate through this list!
    if instrument == 'msi_agm' :  data_stack = np.dstack([mydataset['msi02_agmr'],mydataset['msi03_agmr'],mydataset['msi04_agmr'],mydataset['msi05_agmr'],mydataset['msi06_agmr'],mydataset['msi07_agmr']])
    if instrument == 'oli_agm' :  data_stack = np.dstack([mydataset['oli02_agmr'],mydataset['oli03_agmr'],mydataset['oli04_agmr']])
    
    mydataset = mydataset.assign_coords({dim_name: bandlist})
    mydataset[var_name] = (('time', 'y', dim_name,'x'), data_stack.reshape(mydataset.time.size,mydataset.y.size,np.size(bandlist),mydataset.x.size))
    mydataset = mydataset.transpose('time','y','x',dim_name,...)  #to allow the matrix multiplication the bands dimension needs to be transposed to the end

    #we multiply pixel vector by the full OWT matrix, adding yet another dimension to the dataset (time,x,t,band_name,owt_number are all dimensions)
 
    #bring in the relevant parts of the  msi_owt matrix, and shape to shape of 6 (bands) by 13 (owts) 
    #add a dimension for the OWT type
    owt_dim_values = list(np.array(OWT[instrument]['owt']))   #in practice same for any instrument
    mydataset      = mydataset.assign_coords(owt_num=owt_dim_values)

    #calculate the dot product between each pixel vector and each owt type vector, skipping band1 since we dont have it
    pixel_x_owt_data = np.array(mydataset[var_name]) @ owt_data[:,i:j].T   
   
    #add results back into the dataset
    mydataset[shortname+'_x_owt'] = (('time', 'y', 'x','owt_num'), pixel_x_owt_data) 

    #the dataset now has, for every pixel, the dot product with the spectral reference vector!!
    #but we need the cosines... 
    
    # calculte the self product (the scale) of each of hte OWT reference vectors:
    owt_scale     = np.sum((owt_data[:,i:j] * owt_data[:,i:j]).T,axis=0)**0.5

    # calculate the scale of each pixel vector 
    mydataset[shortname+'_scale']  = ((mydataset[shortname+'_vals'] * mydataset[shortname+'_vals']).sum(dim=(dim_name))) ** 0.5
    
    #now to the crunch...
    mydataset[shortname+'_owt_cosine'] = \
        (mydataset[shortname+'_x_owt'] / mydataset[shortname+'_scale']) / owt_scale

    #now find the owt closest to the msi vector (the largest cosine)
    #to avoid nan problems use np.argmax, then the array is  brought back into the dataset
    mydataset[shortname+'_owt'] = (('time','y','x'),np.argmax(np.array(mydataset[shortname+'_owt_cosine']),axis=3)+1) 

    #but this has zeros instead of nodata:
    mydataset[shortname+'_owt']= xr.where(mydataset[shortname+'_scale']>0,mydataset[shortname+'_owt'],np.nan)

    # REPEAT the analysis on the median for all water areas, by year
    # make an annual median, drop the watermask variable, add the reflectances as a dimension

    median_dataset = mydataset.where(ds['watermask']>0).median(dim=('x','y')).drop_vars('watermask')
    median_dataset = median_dataset.assign_coords({dim_name: bandlist})

    if instrument == 'msi_agm' : data_stack = np.dstack([median_dataset['msi02_agmr'],median_dataset['msi03_agmr'],median_dataset['msi04_agmr'],\
                                                         median_dataset['msi05_agmr'],median_dataset['msi06_agmr'],median_dataset['msi07_agmr']])
    if instrument == 'oli_agm' : data_stack = np.dstack([median_dataset['oli02_agmr'],median_dataset['oli03_agmr'],median_dataset['oli04_agmr']])

    median_dataset[shortname+'_vals'] = (('time', dim_name), data_stack[0,:,:])   #the zero drops the residual dimension leaving time and spectral dim
   
    # dot product of the median msi and oli vectors with the owt vectors:
    median_x_owt_data = np.array(median_dataset[shortname+'_vals']) @ owt_data[:,i:j].T   
    #add results back into the dataset
    median_dataset[shortname+'_x_owt'] = (('time','owt_num'), median_x_owt_data) 
    # scale the dot product to get the cosine of the anle between the ows vectorand the median pixel vector
    median_dataset[shortname+'_vals_scale'] = (median_dataset[shortname+'_vals'] **2).sum(dim=(dim_name))**0.5
    owt_scale = np.sum((owt_data[:,i:j] ** 2).T,axis=0)**0.5
    #now to the crunch...
    median_dataset[shortname+'_owt_cosine'] = \
        (median_dataset[shortname+'_x_owt'] / median_dataset[shortname+'_vals_scale']) / owt_scale
    # find the ows that is closest and save in the median dataset
    median_dataset[shortname+'_owt'] = (('time'),np.argmax(np.array(median_dataset[shortname+'_owt_cosine']),axis=1)+1) 
    #replace zeros with nodata:
    median_dataset[shortname+'_owt']= xr.where(median_dataset[shortname+'_vals_scale']>0,median_dataset[shortname+'_owt'],np.nan)

    #return the pixel-level OWT types, and the overall median results
    return mydataset[shortname+'_owt']
    